# AIMLZG546 – Software Engineering for Machine Learning
## Assignment I – ChurnGuard: Telecom Customer-Churn Prediction & Retention System

**Group No:** `<Fill Group No>`

| Sl. No | BITS ID | Name | Contribution of team member (Qualitative) | % Contribution (out of 100) |
|---|---|---|---|---|
| 1 | | | Requirements & GR4ML Business View | 25 |
| 2 | | | GR4ML Analytics Design & Data Preparation Views | 25 |
| 3 | | | System architecture diagram & pattern selection | 25 |
| 4 | | | Implementation (pipeline + REST service) & testing | 25 |

> Edit the table above with your real names / IDs / contributions before submitting.

**How to run:** `Runtime → Run all` in Google Colab (no extra installs, no dataset upload – data is synthetically generated). Take screenshots of the outputs for your Word/PDF report.

# Objective 1 – Requirements Formulation

## 1. Domain and Problem Statement

**Domain:** Telecommunications (subscription-based mobile & broadband services).

**Problem:** Acquiring a new telecom customer costs 5–7× more than retaining one, yet the company loses ~20 % of its customers every year. Retention teams currently run blanket discount campaigns that are expensive and poorly targeted, and they discover churn only *after* the customer has left.

**Proposed ML application – ChurnGuard:** an ML-based system that ingests CRM, billing and support data, predicts for every active customer the probability of leaving within the next 30 days, and exposes this through a REST service to a retention dashboard. A business-rules layer converts the probability into a recommended retention action (e.g. call + discount for high-risk customers), so that the retention budget is spent only where it matters.

## 2. Requirement Specifications and Measurable Goals

**Functional requirements**

| ID | Requirement |
|---|---|
| FR1 | The system shall ingest customer data (CRM, billing, support) and validate it against a schema. |
| FR2 | The system shall train, evaluate and version a churn-classification model. |
| FR3 | The system shall return a churn probability and a risk band for a single customer on request (REST API). |
| FR4 | The system shall recommend a retention action based on the risk band. |
| FR5 | The system shall log every prediction and expose operational metrics (volume, latency, errors, input drift). |

**Measurable goals (GR4ML goal → indicator → target)**

| Level | Goal | Indicator (KPI) | Target |
|---|---|---|---|
| Business | Reduce customer churn | Monthly churn rate | ≤ 15 % (from ~20 %) within 12 months |
| Business | Reduce retention cost | Cost per retained customer | −25 % |
| Business | Deliver targeted offers | Offer acceptance rate | ≥ 25 % |
| Analytics | Identify customers at risk | ROC-AUC on hold-out set | ≥ 0.80 |
| Analytics | Catch most churners | Recall of churn class | ≥ 0.70 |
| Analytics | Real-time scoring | p95 API latency | ≤ 200 ms |
| Data | Trustworthy input | Schema-valid records | ≥ 99 % |

## 3. GR4ML Views
GR4ML (Goal-oriented Requirements engineering for ML) models an ML solution in several linked views. The notation used in the diagrams below:

| Symbol | Meaning |
|---|---|
| Blue ellipse | Hard **goal** |
| Dashed yellow ellipse | **Soft-goal** (quality goal) |
| Red diamond | **Decision** to be supported |
| Green rounded box | **Indicator / question** (measures a goal) |
| Orange rounded box | **Strategy / task** |
| Purple hexagon | **Analytics algorithm** |
| Lavender box | **Data source / dataset** |

In [ ]:
# ---- common imports and a tiny GR4ML drawing helper -------------------------------------------
import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Polygon, FancyBboxPatch, Rectangle
import numpy as np

STYLE = {  # kind -> (face colour, extra kwargs)
    "goal":      ("#CFE8FF", {}),
    "softgoal":  ("#FFF3C4", {"ls": "--"}),
    "decision":  ("#FFD6D6", {}),
    "indicator": ("#D9F2D9", {}),
    "task":      ("#FDE2C4", {}),
    "algorithm": ("#E3D4F5", {}),
    "data":      ("#E6E6FA", {}),
}

def _draw_node(ax, kind, x, y, w, h, label, fs=8):
    fc, extra = STYLE[kind]
    kw = dict(lw=1.4, ec="#333333", zorder=3, **extra)
    if kind in ("goal", "softgoal"):
        p = Ellipse((x, y), w, h, fc=fc, **kw)
    elif kind == "decision":
        p = Polygon([(x, y + h/2), (x + w/2, y), (x, y - h/2), (x - w/2, y)], closed=True, fc=fc, **kw)
    elif kind == "algorithm":
        d = h * 0.35
        p = Polygon([(x - w/2 + d, y + h/2), (x + w/2 - d, y + h/2), (x + w/2, y),
                     (x + w/2 - d, y - h/2), (x - w/2 + d, y - h/2), (x - w/2, y)], closed=True, fc=fc, **kw)
    elif kind == "data":
        p = Rectangle((x - w/2, y - h/2), w, h, fc=fc, **kw)
    else:
        p = FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.02,rounding_size=0.15", fc=fc, **kw)
    ax.add_patch(p)
    ax.text(x, y, label, ha="center", va="center", fontsize=fs, zorder=4)

def _edge_pt(n, other):
    x, y, w, h = n["x"], n["y"], n["w"], n["h"]
    dx, dy = other["x"] - x, other["y"] - y
    t = 1.0 / max(abs(dx) / (w / 2), abs(dy) / (h / 2), 1e-9)
    return x + dx * t, y + dy * t

def draw_view(nodes, edges, title, size=(14, 7.5), xlim=(0, 14), ylim=(0, 7.5), fname=None):
    fig, ax = plt.subplots(figsize=size)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis("off")
    ax.set_title(title, fontsize=13, fontweight="bold")
    N = {}
    for key, (kind, x, y, w, h, label) in nodes.items():
        N[key] = dict(x=x, y=y, w=w, h=h)
        _draw_node(ax, kind, x, y, w, h, label)
    for e in edges:
        a, b, lab = (e + ("",))[:3]
        p1, p2 = _edge_pt(N[a], N[b]), _edge_pt(N[b], N[a])
        ax.annotate("", xy=p2, xytext=p1, zorder=2, arrowprops=dict(arrowstyle="-|>", lw=1.2, color="#444"))
        if lab:
            ax.text((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2, lab, fontsize=7.5, color="#B00020",
                    ha="center", va="center", zorder=5, bbox=dict(fc="white", ec="none", pad=1))
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=150)
    plt.show()

### 3.1 Business View
Captures *why* the organisation wants the ML solution: goal hierarchy, soft-goals, the decision to be supported, and the indicators that measure success. Arrows read "child → parent" (means-end / contribution).

In [ ]:
business_nodes = {
 "G1":  ("goal",      7.0, 6.5, 3.6, 1.1, "Reduce customer churn"),
 "SG1": ("softgoal",  2.2, 6.5, 3.4, 1.1, "Reduce retention cost"),
 "SG2": ("softgoal", 11.8, 6.5, 3.4, 1.1, "Improve customer\nsatisfaction"),
 "I3":  ("indicator", 2.2, 7.9, 3.4, 0.7, "Cost / retained customer (-25%)"),
 "G3":  ("goal",      3.2, 4.5, 3.6, 1.1, "Deliver targeted\nretention offers"),
 "G2":  ("goal",     10.8, 4.5, 3.6, 1.1, "Identify customers\nat risk of leaving"),
 "I1":  ("indicator", 7.0, 4.5, 3.0, 0.95, "Monthly churn rate\n(target <= 15%)"),
 "S1":  ("task",      1.4, 2.6, 2.6, 1.0, "Strategy: proactive\nretention campaign"),
 "D1":  ("decision",  7.0, 2.6, 4.4, 1.5, "Which customers get\na retention offer?"),
 "I2":  ("indicator", 3.2, 0.7, 3.4, 0.95, "Offer acceptance rate\n(target >= 25%)"),
 "DS":  ("data",      7.0, 0.7, 3.0, 0.95, "CRM + Billing +\nSupport data"),
 "I4":  ("indicator",10.8, 0.7, 3.6, 0.95, "Recall of churners\n(target >= 0.70)"),
}
business_edges = [
 ("G3","G1","AND"), ("G2","G1","AND"), ("G3","SG1","+"), ("G2","SG2","+"),
 ("I1","G1","measures"), ("I3","SG1","measures"),
 ("S1","G3","achieves"), ("D1","G3","supports"), ("D1","G2","supports"),
 ("I2","G3","measures"), ("I4","G2","measures"), ("DS","D1","informs"),
]
draw_view(business_nodes, business_edges, "GR4ML – Business View (ChurnGuard)", ylim=(0, 8.4), size=(14, 8.2), fname="gr4ml_business_view.png")

### 3.2 Analytics Design View
Translates the business goals/decision into *analytics goals*, the algorithms that can achieve them, and the quality soft-goals that drive algorithm selection.

In [ ]:
analytics_nodes = {
 "I1":  ("indicator", 2.3, 6.9, 3.6, 0.95, "Indicator: churn probability\nper customer"),
 "D1":  ("decision",  7.0, 6.9, 4.6, 1.0, "Decision: who gets a retention offer?"),
 "AG":  ("goal",      7.0, 5.1, 5.8, 1.4, "Analytics goal: PREDICT whether a customer\nwill churn within 30 days\n(binary classification)"),
 "SG3": ("softgoal",  2.5, 3.4, 3.4, 1.0, "Interpretability"),
 "SG2": ("softgoal",  7.0, 3.4, 3.4, 1.0, "Low latency\np95 <= 200 ms"),
 "SG1": ("softgoal", 11.5, 3.4, 3.8, 1.0, "Prediction quality\nROC-AUC>=0.80, Recall>=0.70"),
 "A1":  ("algorithm", 2.5, 1.7, 3.0, 1.0, "Logistic Regression"),
 "A2":  ("algorithm", 7.0, 1.7, 3.0, 1.0, "Random Forest"),
 "A3":  ("algorithm",11.5, 1.7, 3.0, 1.0, "Gradient Boosting"),
 "T1":  ("task",      7.0, 0.35, 9.0, 0.6, "Selection: 3-fold CV (ROC-AUC) -> best model, threshold tuned on F2"),
}
analytics_edges = [
 ("I1","D1","informs"), ("AG","D1","supports"),
 ("SG3","AG",""), ("SG2","AG",""), ("SG1","AG",""),
 ("A1","SG3","++"), ("A1","SG2","+"), ("A2","SG2","-"), ("A2","SG1","+"), ("A3","SG1","++"),
 ("A1","T1",""), ("A2","T1",""), ("A3","T1",""),
]
draw_view(analytics_nodes, analytics_edges, "GR4ML – Analytics Design View (ChurnGuard)", ylim=(0, 7.8), size=(14, 7.8), fname="gr4ml_analytics_view.png")

### 3.3 Data Preparation View
Shows the data sources, the preparation tasks that must be applied (in order), and the resulting datasets.

In [ ]:
data_nodes = {
 "S1": ("data", 1.6, 6.0, 2.6, 0.9, "CRM\n(demographics, contract)"),
 "S2": ("data", 1.6, 4.4, 2.6, 0.9, "Billing DB\n(charges, payments)"),
 "S3": ("data", 1.6, 2.8, 2.6, 0.9, "Support system\n(tickets, calls)"),
 "T0": ("task", 4.7, 4.4, 2.2, 1.0, "Extract & join\nby customer_id"),
 "T1": ("task", 7.3, 4.4, 2.3, 1.0, "Validate schema,\nranges, duplicates"),
 "T2": ("task", 9.9, 4.4, 2.3, 1.0, "Clean: impute\nmissing values"),
 "T3": ("task", 12.5, 4.4, 2.3, 1.0, "Feature engineering\n(avg spend, new, ...)"),
 "T5": ("task", 12.5, 2.3, 2.3, 1.0, "Stratified\ntrain/test split"),
 "T4": ("task", 9.9, 2.3, 2.5, 1.0, "Encode + scale\n(fit on train only)"),
 "DS": ("data", 7.0, 0.7, 4.8, 0.9, "Output: model-ready training & test datasets"),
}
data_edges = [("S1","T0"),("S2","T0"),("S3","T0"),("T0","T1"),("T1","T2"),("T2","T3"),("T3","T5"),("T5","T4"),("T4","DS")]
draw_view(data_nodes, data_edges, "GR4ML – Data Preparation View (ChurnGuard)", ylim=(0, 7), size=(14, 7), fname="gr4ml_data_view.png")

## 4. Top-3 Quality Requirements and Justification

| # | Quality attribute (ISO/IEC 25010 / ML) | Measurable requirement | Justification |
|---|---|---|---|
| 1 | **Predictive performance (functional suitability / accuracy)** | ROC-AUC ≥ 0.80 and recall of churners ≥ 0.70 on a hold-out set | The whole business value comes from correctly ranking customers by risk. A missed churner (false negative) costs the full lifetime value of the customer, while a false positive only costs one unnecessary offer – therefore **recall** is prioritised and ROC-AUC is used to compare models independent of threshold. |
| 2 | **Performance efficiency (latency & scalability)** | p95 response time ≤ 200 ms per prediction; stateless service that scales horizontally | Agents use the prediction while a customer is on the line / on the app, so scoring must feel instantaneous. Slow or unavailable scoring would make the ML output useless for real-time retention decisions. |
| 3 | **Maintainability (modularity, modifiability, testability, monitorability)** | Each processing step is an independent, unit-testable component; model can be replaced without changing the API; every prediction is logged and drift is observable | Customer behaviour drifts (new plans, competitors), so models must be retrained and replaced regularly. A modular pipeline and decoupled serving layer keep the cost of change low – this is a core SE4ML concern ("hidden technical debt in ML systems"). |

*Why not others?* Security/privacy matter (PII) but the demo works on pseudonymised data; fairness is monitored but is secondary to the three above for this use-case.

# Objective 2 – System Architecture

## 5. System Architecture Diagram
ML components are shown in **blue**, non-ML (traditional software / infrastructure) components in **green/grey**.

In [ ]:
def box(ax, x, y, w, h, text, ml, fs=8):
    fc = "#BBDDF8" if ml else "#DDEFD2"
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                fc=fc, ec="#222", lw=1.3, zorder=3))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fs, zorder=4)
    return dict(l=(x, y+h/2), r=(x+w, y+h/2), t=(x+w/2, y+h), b=(x+w/2, y))

def arrow(ax, p, q, text="", style="-|>", ls="-", off=(0, 0)):
    ax.annotate("", xy=q, xytext=p, zorder=2, arrowprops=dict(arrowstyle=style, lw=1.2, color="#333", ls=ls))
    if text:
        ax.text((p[0]+q[0])/2 + off[0], (p[1]+q[1])/2 + off[1], text, fontsize=7, color="#B00020",
                ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.8), zorder=5)

fig, ax = plt.subplots(figsize=(16, 9.5)); ax.set_xlim(0, 16); ax.set_ylim(0, 9.5); ax.axis("off")
ax.set_title("ChurnGuard – System Architecture (ML vs non-ML components)", fontsize=14, fontweight="bold")

# swim-lane backgrounds
ax.add_patch(Rectangle((0.2, 5.0), 15.6, 4.1, fc="#F7F7F7", ec="#888", ls="--", zorder=0))
ax.text(0.35, 8.85, "OFFLINE TRAINING PIPELINE  (Pattern 1: Pipes & Filters)", fontsize=9, fontweight="bold", va="top")
ax.add_patch(Rectangle((0.2, 0.2), 15.6, 4.6, fc="#FBF8EE", ec="#888", ls="--", zorder=0))
ax.text(0.35, 4.6, "ONLINE SERVING  (Pattern 2: Model-as-a-Service / REST microservice)", fontsize=9, fontweight="bold", va="top")

# offline
src = box(ax, 0.5, 6.0, 2.0, 1.8, "Data sources\nCRM | Billing |\nSupport tickets", False)
ing = box(ax, 3.1, 6.0, 1.9, 1.8, "Ingestion &\nValidation filter\n(schema, range,\nduplicates)", False)
cln = box(ax, 5.5, 6.0, 1.9, 1.8, "Cleaning filter\n(impute missing)", True)
fe  = box(ax, 7.9, 6.0, 1.9, 1.8, "Feature-\nengineering\nfilter", True)
trn = box(ax, 10.3, 6.0, 1.9, 1.8, "Model training\n(LR / RF / GB,\nCV selection)", True)
evl = box(ax, 12.7, 6.0, 2.8, 1.8, "Evaluation & threshold\ntuning (ROC-AUC,\nrecall, F2)", True)
reg = box(ax, 12.7, 5.2, 2.8, 0.6, "Model registry (versioned artefact)", False, fs=7.5)
for a, b in [(src["r"], ing["l"]), (ing["r"], cln["l"]), (cln["r"], fe["l"]), (fe["r"], trn["l"]), (trn["r"], evl["l"])]:
    arrow(ax, a, b)
arrow(ax, evl["b"], (evl["b"][0], 5.8))

# online
cli = box(ax, 0.5, 2.3, 2.2, 1.7, "Client apps\nRetention dashboard /\nCall-centre CRM", False)
gw  = box(ax, 3.4, 2.3, 2.0, 1.7, "REST API /\nAPI gateway\n(Flask)\nauth + validation", False)
inf = box(ax, 6.1, 2.3, 2.3, 1.7, "Prediction service\n(loads model from registry,\nreuses same filters)", True)
rul = box(ax, 9.1, 2.3, 2.1, 1.7, "Business-rules\nengine\n(risk band -> action)", False)
log = box(ax, 11.9, 2.3, 1.9, 1.7, "Prediction log &\nmetrics store", False)
mon = box(ax, 11.9, 0.45, 3.6, 1.2, "Monitoring: latency, errors,\ninput-drift check (ML)\n-> triggers retraining", True)
arrow(ax, cli["r"], gw["l"], "JSON request", off=(0, 0.28))
arrow(ax, gw["r"], inf["l"], "validated features", off=(0, 0.28))
arrow(ax, inf["r"], rul["l"], "probability", off=(0, 0.28))
arrow(ax, rul["r"], log["l"], "action + log", off=(0, 0.28))
ax.plot([rul["b"][0], rul["b"][0], cli["b"][0]], [2.3, 1.2, 1.2], color="#333", lw=1.2, zorder=1)   # return path
arrow(ax, (cli["b"][0], 1.2), (cli["b"][0], 2.3))
ax.text(5.5, 1.0, "JSON response: probability, risk band, recommended action", fontsize=7.5, color="#B00020", ha="center")
arrow(ax, log["b"], (log["b"][0], 1.65))
ax.plot([15.0, 15.0, 11.25], [1.65, 4.95, 4.95], color="#333", lw=1.2, ls="--", zorder=1)   # retrain trigger
arrow(ax, (11.25, 4.95), (11.25, 6.0), ls="--")
ax.text(13.3, 4.95, "retrain trigger", fontsize=7, color="#B00020", ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.8), zorder=5)
arrow(ax, (reg["b"][0]-0.8, 5.2), (inf["t"][0]+0.5, 4.0), "load model v1", ls="--", off=(1.2, 0.25))

# legend
ax.add_patch(Rectangle((0.5, 0.35), 0.35, 0.3, fc="#BBDDF8", ec="#222")); ax.text(0.95, 0.5, "ML component", fontsize=8, va="center")
ax.add_patch(Rectangle((2.6, 0.35), 0.35, 0.3, fc="#DDEFD2", ec="#222")); ax.text(3.05, 0.5, "Non-ML component", fontsize=8, va="center")
plt.tight_layout(); plt.savefig("system_architecture.png", dpi=150); plt.show()

## 6. Selection of Architectural Patterns

| | Pattern | Where applied | Quality attribute supported | Rationale |
|---|---|---|---|---|
| **P1** | **Pipes and Filters** (pipeline) | Data validation → cleaning → feature engineering → training pipeline, **and** re-used at inference time | Maintainability, testability, reuse; training-serving consistency (→ prediction quality) | Each transformation is an independent filter with a uniform interface (`fit` / `process`). Filters can be unit-tested, replaced or re-ordered without touching others, and the *same* fitted filters are re-used when serving, preventing training/serving skew – one of the most common ML bugs. |
| **P2** | **Model-as-a-Service** (client-server / microservice with REST API, a.k.a. "Wrapper/Service" pattern for ML) | Prediction service that wraps the trained model and exposes `/predict`, `/health`, `/metrics` | Performance efficiency, scalability, modifiability, interoperability | The model is deployed as a stateless service behind a stable JSON contract. Clients (dashboard, CRM) are decoupled from the ML technology, the model can be swapped by loading a new registry version, and the service can be scaled horizontally to meet the p95 ≤ 200 ms goal. |

## 7. Implementation
Technology stack: **Python, pandas, scikit-learn** (ML pipeline), **joblib** (model registry/serialisation), **Flask** (REST service), **requests** (client), **matplotlib** (diagrams/plots). All runs inside Colab.

### 7.1 Synthetic data (stands in for the CRM + Billing + Support extract)

In [ ]:
import pandas as pd, numpy as np, time, json, threading, joblib, os, logging
from datetime import datetime

rng = np.random.default_rng(42)

def make_customers(n=6000, seed=42):
    r = np.random.default_rng(seed)
    tenure   = r.integers(1, 73, n)
    contract = r.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.20])
    internet = r.choice(["DSL", "Fiber", "None"], n, p=[0.35, 0.45, 0.20])
    payment  = r.choice(["Credit card", "Bank transfer", "Electronic check", "Mailed check"], n, p=[0.3, 0.25, 0.3, 0.15])
    monthly  = np.round(np.clip(r.normal(65, 25, n) + 15 * (internet == "Fiber") - 25 * (internet == "None"), 18, 120), 2)
    calls    = r.poisson(1.5, n)
    senior   = r.binomial(1, 0.16, n)
    paperless= r.binomial(1, 0.6, n)
    total    = np.round(tenure * monthly * r.normal(1.0, 0.03, n), 2)
    logit = (-2.6 + 1.6 * (contract == "Month-to-month") - 1.0 * (contract == "Two year")
             - 0.045 * tenure + 0.022 * (monthly - 65) + 0.50 * calls
             + 0.70 * (internet == "Fiber") + 0.70 * (payment == "Electronic check") + 0.30 * senior)
    churn = r.binomial(1, 1 / (1 + np.exp(-logit)))
    df = pd.DataFrame(dict(customer_id=[f"C{i:05d}" for i in range(n)], tenure=tenure, contract=contract,
                           internet_service=internet, payment_method=payment, monthly_charges=monthly,
                           total_charges=total, support_calls=calls, senior_citizen=senior,
                           paperless_billing=paperless, churn=churn))
    # inject realistic data-quality problems
    df.loc[r.choice(n, 90, replace=False), "total_charges"] = np.nan
    df.loc[r.choice(n, 40, replace=False), "monthly_charges"] = np.nan
    df = pd.concat([df, df.sample(25, random_state=1)], ignore_index=True)   # duplicates
    return df

raw = make_customers()
print("Raw shape:", raw.shape, "| churn rate: %.1f%%" % (100 * raw.churn.mean()))
print("Missing values:\n", raw.isna().sum()[raw.isna().sum() > 0])
raw.head()

### 7.2 Pattern 1 – Pipes and Filters
Every filter implements the same interface; the `PipesAndFilters` runner connects them (the "pipes"). Training calls `fit_process`, serving calls `process` with the *already fitted* filters.

In [ ]:
class DataValidationError(ValueError):
    pass

class Filter:
    """Common filter interface (the 'filter' contract of the Pipes & Filters pattern)."""
    name = "filter"
    def fit(self, df):        return self          # learn parameters (if any) from training data
    def process(self, df):    raise NotImplementedError

class ValidationFilter(Filter):
    name = "validation"
    REQUIRED = ["tenure", "contract", "internet_service", "payment_method", "monthly_charges",
                "total_charges", "support_calls", "senior_citizen", "paperless_billing"]
    CATEGORIES = {"contract": {"Month-to-month", "One year", "Two year"},
                  "internet_service": {"DSL", "Fiber", "None"},
                  "payment_method": {"Credit card", "Bank transfer", "Electronic check", "Mailed check"}}
    RANGES = {"tenure": (0, 120), "monthly_charges": (0, 500), "support_calls": (0, 100)}

    def process(self, df):
        missing = [c for c in self.REQUIRED if c not in df.columns]
        if missing:
            raise DataValidationError(f"missing columns: {missing}")
        for col, allowed in self.CATEGORIES.items():
            bad = set(df[col].dropna().unique()) - allowed
            if bad:
                raise DataValidationError(f"invalid values for '{col}': {sorted(bad)}")
        for col, (lo, hi) in self.RANGES.items():
            vals = df[col].dropna()
            if ((vals < lo) | (vals > hi)).any():
                raise DataValidationError(f"'{col}' out of range [{lo}, {hi}]")
        if "customer_id" in df.columns:
            df = df.drop_duplicates(subset="customer_id")
        return df.copy()

class CleaningFilter(Filter):
    name = "cleaning"
    def fit(self, df):
        self.median_monthly_ = float(df["monthly_charges"].median())
        return self
    def process(self, df):
        df = df.copy()
        df["monthly_charges"] = df["monthly_charges"].fillna(self.median_monthly_)
        df["total_charges"]   = df["total_charges"].fillna(df["tenure"] * df["monthly_charges"])
        return df

class FeatureEngineeringFilter(Filter):
    name = "feature_engineering"
    def process(self, df):
        df = df.copy()
        df["avg_monthly_spend"]  = df["total_charges"] / df["tenure"].clip(lower=1)
        df["is_new_customer"]    = (df["tenure"] < 12).astype(int)
        df["high_support_calls"] = (df["support_calls"] >= 3).astype(int)
        df["charge_vs_avg"]      = df["monthly_charges"] - df["avg_monthly_spend"]
        return df

class PipesAndFilters:
    """The 'pipes': pushes the output of one filter into the next."""
    def __init__(self, filters, verbose=True):
        self.filters, self.verbose = filters, verbose
    def _log(self, f, n_in, n_out):
        if self.verbose:
            print(f"  [filter] {f.name:<20} rows in={n_in:>5}  rows out={n_out:>5}")
    def fit_process(self, df):
        for f in self.filters:
            n = len(df); df = f.fit(df).process(df); self._log(f, n, len(df))
        return df
    def process(self, df):
        for f in self.filters:
            n = len(df); df = f.process(df); self._log(f, n, len(df))
        return df

filters = PipesAndFilters([ValidationFilter(), CleaningFilter(), FeatureEngineeringFilter()])

# --- unit-test style demonstration of testability (quality requirement #3) -----------------------
bad = raw.head(3).drop(columns=["contract"])
try: ValidationFilter().process(bad)
except DataValidationError as e: print("Validation filter correctly rejected bad input ->", e)

print("\nRunning training-time pipeline:")
clean = filters.fit_process(raw)
print("\nColumns after pipeline:", list(clean.columns))
print("Remaining missing values:", int(clean.isna().sum().sum()))

### 7.3 Model training, selection and evaluation (final stage of the pipeline)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, recall_score, precision_score, f1_score, fbeta_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report)

NUM = ["tenure", "monthly_charges", "total_charges", "support_calls", "senior_citizen", "paperless_billing",
       "avg_monthly_spend", "is_new_customer", "high_support_calls", "charge_vs_avg"]
CAT = ["contract", "internet_service", "payment_method"]
FEATURES = NUM + CAT

def make_model(clf):
    pre = ColumnTransformer([("num", StandardScaler(), NUM),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("prep", pre), ("clf", clf)])

X, y = clean[FEATURES], clean["churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_tr.shape}  Test: {X_te.shape}  churn rate train/test: {y_tr.mean():.3f}/{y_te.mean():.3f}")

candidates = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest":       RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1),
    "GradientBoosting":   GradientBoostingClassifier(random_state=42),
}
cv = StratifiedKFold(3, shuffle=True, random_state=42)
cv_scores = {}
for name, clf in candidates.items():
    s = cross_val_score(make_model(clf), X_tr, y_tr, cv=cv, scoring="roc_auc")
    cv_scores[name] = s.mean(); print(f"{name:<20} CV ROC-AUC = {s.mean():.4f} (+/- {s.std():.4f})")

best_name = max(cv_scores, key=cv_scores.get)
print("\n>> Selected model:", best_name)
best = make_model(candidates[best_name])

# threshold tuning on out-of-fold predictions (favours recall -> F2), not on the test set
oof = cross_val_predict(best, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
grid = np.linspace(0.1, 0.7, 61)
THRESHOLD = float(grid[np.argmax([fbeta_score(y_tr, oof >= t, beta=2) for t in grid])])
print(f">> Decision threshold (max F2 on out-of-fold data): {THRESHOLD:.2f}")

best.fit(X_tr, y_tr)
proba = best.predict_proba(X_te)[:, 1]; pred = (proba >= THRESHOLD).astype(int)
metrics = dict(roc_auc=roc_auc_score(y_te, proba), recall=recall_score(y_te, pred),
               precision=precision_score(y_te, pred), f1=f1_score(y_te, pred))
print("\nHold-out metrics:", {k: round(v, 3) for k, v in metrics.items()})
print(classification_report(y_te, pred, target_names=["stay", "churn"]))

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_te, proba, ax=axs[0]); axs[0].set_title("ROC curve (test set)")
ConfusionMatrixDisplay(confusion_matrix(y_te, pred), display_labels=["stay", "churn"]).plot(ax=axs[1], colorbar=False)
axs[1].set_title(f"Confusion matrix (threshold={THRESHOLD:.2f})"); plt.tight_layout(); plt.savefig("evaluation.png", dpi=150); plt.show()

In [ ]:
# ---- Model registry: versioned artefact = fitted filters + model + threshold + metadata -----------
REGISTRY_DIR = "model_registry"; os.makedirs(REGISTRY_DIR, exist_ok=True)
artefact = dict(version="v1", trained_at=datetime.now().isoformat(timespec="seconds"), model_name=best_name,
                filters=filters, model=best, threshold=THRESHOLD, features=FEATURES, metrics=metrics,
                train_means={"monthly_charges": float(X_tr["monthly_charges"].mean()),
                             "tenure": float(X_tr["tenure"].mean())})
joblib.dump(artefact, f"{REGISTRY_DIR}/churn_v1.joblib")
print("Saved ->", f"{REGISTRY_DIR}/churn_v1.joblib", "| size: %.1f KB" % (os.path.getsize(f"{REGISTRY_DIR}/churn_v1.joblib") / 1024))

### 7.4 Pattern 2 – Model-as-a-Service (REST microservice)
The prediction service is a stateless Flask app. It loads the versioned artefact from the registry at start-up, **re-uses the same Pipes-and-Filters pipeline** for incoming records, applies the business rules and logs every call. It runs in a background thread inside Colab so we can call it like any remote client.

In [ ]:
from flask import Flask, request, jsonify
from werkzeug.serving import make_server

# ---------- non-ML component: business-rules engine ----------
def recommend_action(p, threshold):
    if p >= 0.60:       return "HIGH",   "Agent call + 20% loyalty discount"
    if p >= threshold:  return "MEDIUM", "Send personalised plan-upgrade offer"
    return "LOW", "No action"

# ---------- non-ML component: model registry loader ----------
def load_model(version="v1"):
    return joblib.load(f"{REGISTRY_DIR}/churn_{version}.joblib")

def create_app(model_version="v1"):
    art = load_model(model_version)
    serving_filters = PipesAndFilters(art["filters"].filters, verbose=False)
    app = Flask("churnguard")
    app.config["ART"] = art
    stats = dict(requests=0, errors=0, latencies=[], bands={"HIGH": 0, "MEDIUM": 0, "LOW": 0}, mc_sum=0.0)
    log = logging.getLogger("prediction_log"); log.setLevel(logging.INFO); log.handlers.clear()
    h = logging.FileHandler("predictions.log"); h.setFormatter(logging.Formatter("%(asctime)s %(message)s")); log.addHandler(h)

    @app.get("/health")
    def health():
        return jsonify(status="ok", model_version=art["version"], model=art["model_name"])

    @app.post("/predict")
    def predict():
        t0 = time.perf_counter(); stats["requests"] += 1
        try:
            payload = request.get_json(force=True)
            records = payload if isinstance(payload, list) else [payload]
            df = pd.DataFrame(records)
            df = serving_filters.process(df)                       # <- Pattern 1 re-used at inference time
            proba = art["model"].predict_proba(df[art["features"]])[:, 1]
            out = []
            for i, p in enumerate(proba):
                band, action = recommend_action(float(p), art["threshold"])
                stats["bands"][band] += 1
                out.append(dict(customer_id=records[i].get("customer_id"), churn_probability=round(float(p), 4),
                                churn_prediction=int(p >= art["threshold"]), risk_band=band, recommended_action=action))
                log.info(json.dumps(dict(customer_id=records[i].get("customer_id"), p=round(float(p), 4), band=band)))
            stats["mc_sum"] += float(df["monthly_charges"].mean())
            ms = (time.perf_counter() - t0) * 1000; stats["latencies"].append(ms)
            return jsonify(model_version=art["version"], predictions=out, latency_ms=round(ms, 2))
        except DataValidationError as e:
            stats["errors"] += 1; return jsonify(error="validation_error", detail=str(e)), 400
        except Exception as e:
            stats["errors"] += 1; return jsonify(error="server_error", detail=str(e)), 500

    @app.get("/metrics")
    def metrics_ep():
        lat = np.array(stats["latencies"]) if stats["latencies"] else np.array([0.0])
        n_ok = len(stats["latencies"])
        drift = None
        if n_ok:
            live = stats["mc_sum"] / n_ok
            drift = round(abs(live - art["train_means"]["monthly_charges"]) / art["train_means"]["monthly_charges"], 4)
        return jsonify(requests=stats["requests"], errors=stats["errors"], p50_ms=round(float(np.percentile(lat, 50)), 2),
                       p95_ms=round(float(np.percentile(lat, 95)), 2), risk_band_counts=stats["bands"],
                       monthly_charges_relative_drift=drift, drift_alert=bool(drift is not None and drift > 0.15))
    return app

class ServerThread(threading.Thread):
    def __init__(self, app, port=5055):
        super().__init__(daemon=True); self.srv = make_server("127.0.0.1", port, app, threaded=True)
    def run(self): self.srv.serve_forever()
    def stop(self): self.srv.shutdown()

logging.getLogger("werkzeug").setLevel(logging.ERROR)   # keep the notebook output clean
if os.path.exists("predictions.log"): os.remove("predictions.log")
server = ServerThread(create_app("v1")); server.start(); time.sleep(1)
BASE = "http://127.0.0.1:5055"
print("Service started at", BASE)

### 7.5 Calling the service as a client (take screenshots of these outputs)

In [ ]:
import requests
print("GET /health ->", requests.get(f"{BASE}/health").json(), "\n")

customer_high = dict(customer_id="NEW-001", tenure=3, contract="Month-to-month", internet_service="Fiber",
                     payment_method="Electronic check", monthly_charges=95.5, total_charges=280.0,
                     support_calls=5, senior_citizen=1, paperless_billing=1)
customer_low  = dict(customer_id="NEW-002", tenure=60, contract="Two year", internet_service="DSL",
                     payment_method="Credit card", monthly_charges=45.0, total_charges=2700.0,
                     support_calls=0, senior_citizen=0, paperless_billing=0)

for c in (customer_high, customer_low):
    r = requests.post(f"{BASE}/predict", json=c); print("POST /predict", c["customer_id"], "->", r.status_code)
    print(json.dumps(r.json(), indent=2), "\n")

print("Batch request (3 customers from the hold-out set):")
sample = raw.drop(columns="churn").drop_duplicates("customer_id").sample(3, random_state=7).to_dict(orient="records")
sample = [{k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in s.items()} for s in sample]
print(json.dumps(requests.post(f"{BASE}/predict", json=sample).json()["predictions"], indent=2))

print("\nInvalid request (unknown contract type) ->")
bad = dict(customer_high, contract="Weekly")
r = requests.post(f"{BASE}/predict", json=bad); print(r.status_code, r.json())

### 7.6 Verifying the quality requirements (load test + monitoring endpoint)

In [ ]:
# ---- load test: 300 sequential calls ----
pool = raw.drop(columns="churn").drop_duplicates("customer_id").sample(300, random_state=3)
pool = [{k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in row.items()} for row in pool.to_dict(orient="records")]
client_lat = []
for rec in pool:
    t0 = time.perf_counter(); requests.post(f"{BASE}/predict", json=rec); client_lat.append((time.perf_counter() - t0) * 1000)

m = requests.get(f"{BASE}/metrics").json()
print("GET /metrics ->", json.dumps(m, indent=2))

plt.figure(figsize=(6, 3.5)); plt.hist(client_lat, bins=30, color="#4C9BE8")
plt.axvline(np.percentile(client_lat, 95), color="r", ls="--", label=f"p95 = {np.percentile(client_lat, 95):.1f} ms")
plt.axvline(200, color="k", ls=":", label="target 200 ms"); plt.xlabel("end-to-end latency (ms)"); plt.legend()
plt.title("Client-side latency of /predict"); plt.tight_layout(); plt.savefig("latency.png", dpi=150); plt.show()

# ---- requirement verification table ----
rows = [("Predictive perf. – ROC-AUC", ">= 0.80", f"{metrics['roc_auc']:.3f}", metrics["roc_auc"] >= 0.80),
        ("Predictive perf. – Recall (churn)", ">= 0.70", f"{metrics['recall']:.3f}", metrics["recall"] >= 0.70),
        ("Performance – p95 latency (ms)", "<= 200", f"{np.percentile(client_lat, 95):.1f}", np.percentile(client_lat, 95) <= 200),
        ("Maintainability – model swap w/o API change", "yes", "yes (registry version)", True),
        ("Maintainability – prediction logging", "yes", f"{sum(1 for _ in open('predictions.log'))} log lines", True)]
res = pd.DataFrame(rows, columns=["Requirement", "Target", "Measured", "Met?"]); res["Met?"] = res["Met?"].map({True: "PASS", False: "FAIL"})
display(res) if "display" in globals() else print(res.to_string(index=False))
print("\nFirst lines of predictions.log:"); print("".join(open("predictions.log").readlines()[:4]))

In [ ]:
# ---- Demonstrating Maintainability: hot-swap the model version without changing the API contract ----
# (e.g. retrain a challenger, save it as v2 and restart the service with create_app("v2"))
challenger = make_model(GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, random_state=1)).fit(X_tr, y_tr)
art2 = dict(artefact, version="v2", model_name="GradientBoosting-challenger", model=challenger)
joblib.dump(art2, f"{REGISTRY_DIR}/churn_v2.joblib")
server.stop(); server = ServerThread(create_app("v2")); server.start(); time.sleep(1)
print("After swap:", requests.get(f"{BASE}/health").json())
print(requests.post(f"{BASE}/predict", json=customer_high).json()["predictions"])
server.stop()

## 8. Summary
* **Objective 1:** problem statement, FR/NFR, measurable goals, three GR4ML views (Business, Analytics Design, Data Preparation) and top-3 quality requirements (predictive performance, performance efficiency, maintainability) with justification.
* **Objective 2:** layered architecture diagram separating ML and non-ML components; two patterns applied – **Pipes & Filters** (training and inference pipeline) and **Model-as-a-Service** (REST microservice with business rules, logging and monitoring) – and both implemented and verified in this notebook against the stated quality requirements.

### Group-member contribution (final)
*(Edit with your real details – must match the table at the top.)*
| Member | Work done | % |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |